# 04 Tuning: feature-group ablation (Day 8)

Question: how much does each engineered feature group change ROC-AUC and PR-AUC?

Method:
- Each group is tested two ways: **added alone to the base set** and **removed alone from the full set**.
- Identical stratified 5-fold CV every time, so differences are **paired by fold**.
- Two models (Logistic Regression, Random Forest), run **with and without `error_code`**.
- Only the training split is used. The test split is never touched here.
- A delta only counts as real when its mean is larger than the std of the per-fold differences.

In [1]:
import os
import sys

# Run from the repo root so `src` and `data/` resolve, whether the notebook
# is started from the repo root or from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print(os.getcwd())

d:\ml-engineer-internship


In [2]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.model_selection import StratifiedKFold, cross_validate  # noqa: E402

from src.features import HISTORY_FEATURES, RATIO_FEATURES, TIME_FEATURES  # noqa: E402
from src.pipeline import (  # noqa: E402
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    SEED,
    build_pipeline,
    load_splits,
    split_xy,
)

## Data (training split only)

In [3]:
train, _val, _test = load_splits(with_features=True)
ENGINEERED = RATIO_FEATURES + TIME_FEATURES + HISTORY_FEATURES
x_train, y_train = split_xy(train, extra=ENGINEERED + ["account_id"])
print(x_train.shape, round(y_train.mean(), 4))

(17500, 21) 0.0391


## CV and models

These must match `03_baselines.ipynb` (same fold seed, same model settings) so results are comparable with Day 7. Edit this cell if Day 7 used different settings.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
MODELS = {
    "logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
    "rf": lambda: RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
}
SCORING = {"roc": "roc_auc", "pr": "average_precision"}

## Feature groups and pipeline config

In [5]:
GROUPS = {
    "ratio": {"num": RATIO_FEATURES},
    "time": {"num": TIME_FEATURES},
    "history": {"num": HISTORY_FEATURES},
    "freq": {"freq": ["account_id"]},
    "target": {"target": ["account_id"]},
    "bins": {"bin": ["latency_ms", "retry_count"]},
}


def pick(groups, key):
    """Collect the columns of one kind from the chosen groups (None if empty)."""
    cols = [c for g in groups for c in GROUPS[g].get(key, [])]
    return cols or None


def make_kwargs(groups, use_error_code):
    """Translate a set of feature groups into build_pipeline arguments."""
    cat = [c for c in CATEGORICAL_FEATURES if use_error_code or c != "error_code"]
    num = list(NUMERIC_FEATURES)
    for g in groups:
        num += GROUPS[g].get("num", [])
    return {
        "num_cols": num,
        "cat_cols": cat,
        "freq_cols": pick(groups, "freq"),
        "target_cols": pick(groups, "target"),
        "bin_cols": pick(groups, "bin"),
    }


def fold_scores(model_name, groups, use_error_code):
    """Per-fold ROC-AUC and PR-AUC on the same folds every time."""
    pipe = build_pipeline(MODELS[model_name](), **make_kwargs(groups, use_error_code))
    res = cross_validate(pipe, x_train, y_train, cv=cv, scoring=SCORING)
    return {"roc": res["test_roc"], "pr": res["test_pr"]}

## Run the ablation (takes a few minutes)

In [6]:
rows = []
all_groups = list(GROUPS)
for use_ec in (True, False):
    for model in MODELS:
        base = fold_scores(model, [], use_ec)
        full = fold_scores(model, all_groups, use_ec)
        cases = [("ALL", "add_to_base", full, base)]
        for g in all_groups:
            added = fold_scores(model, [g], use_ec)
            dropped = fold_scores(model, [x for x in all_groups if x != g], use_ec)
            cases.append((g, "add_to_base", added, base))
            cases.append((g, "drop_from_full", full, dropped))
        for group, mode, with_g, without_g in cases:
            for metric in ("roc", "pr"):
                diff = with_g[metric] - without_g[metric]  # paired by fold
                rows.append(
                    {
                        "error_code": use_ec,
                        "model": model,
                        "group": group,
                        "mode": mode,
                        "metric": metric,
                        "mean": diff.mean(),
                        "std": diff.std(ddof=1),
                    }
                )
results = pd.DataFrame(rows)
results["verdict"] = np.select(
    [results["mean"] > results["std"], results["mean"] < -results["std"]],
    ["helps", "hurts"],
    default="no clear effect",
)

## Ablation table

Values are the mean change in the metric (positive means the group helps). `add_to_base` = group added alone to the base features. `drop_from_full` = full set minus that group.

In [7]:
os.makedirs("reports", exist_ok=True)
results.round(4).to_csv("reports/ablation_results.csv", index=False)
table = results.pivot_table(
    index=["error_code", "model", "group"],
    columns=["mode", "metric"],
    values="mean",
).round(4)
table

mode                      add_to_base         drop_from_full        
metric                             pr     roc             pr     roc
error_code model  group                                             
False      logreg ALL          0.0034 -0.0015            NaN     NaN
                  bins         0.0041  0.0007         0.0050  0.0000
                  freq         0.0000 -0.0009        -0.0002 -0.0009
                  history     -0.0006 -0.0014        -0.0003 -0.0006
                  ratio       -0.0006 -0.0009        -0.0003 -0.0013
                  target       0.0009  0.0009         0.0009  0.0004
                  time        -0.0006  0.0006        -0.0004  0.0004
           rf     ALL          0.0103  0.0131            NaN     NaN
                  bins        -0.0055 -0.0010         0.0044  0.0024
                  freq         0.0004  0.0072         0.0002 -0.0034
                  history     -0.0001  0.0000        -0.0020 -0.0023
                  ratio        0.0045  0.0027        -0.0005  0.0039
                  target       0.0052  0.0141         0.0005 -0.0041
                  time         0.0114  0.0130        -0.0011  0.0084
True       logreg ALL         -0.0057 -0.0056            NaN     NaN
                  bins         0.0007 -0.0036         0.0002 -0.0028
                  freq        -0.0004 -0.0006        -0.0007 -0.0006
                  history     -0.0017 -0.0019        -0.0006 -0.0016
                  ratio       -0.0021 -0.0001        -0.0022 -0.0001
                  target       0.0007  0.0003         0.0004  0.0010
                  time        -0.0021 -0.0005        -0.0023 -0.0007
           rf     ALL          0.0095 -0.0070            NaN     NaN
                  bins        -0.0058 -0.0058        -0.0094 -0.0037
                  freq         0.0151  0.0050        -0.0045 -0.0004
                  history      0.0099 -0.0088        -0.0038 -0.0076
                  ratio        0.0054  0.0076        -0.0029 -0.0013
                  target       0.0107  0.0053         0.0003 -0.0023
                  time         0.0061 -0.0084        -0.0122 -0.0048

## PR-AUC verdicts

`helps` / `hurts` only when the mean delta is larger than the std of the per-fold differences. Otherwise `no clear effect`.

In [8]:
results[results["metric"] == "pr"].round(4).sort_values(
    ["error_code", "model", "mode", "mean"], ascending=[True, True, True, False]
)

,error_code,model,group,mode,metric,mean,std,verdict
75,False,logreg,bins,add_to_base,pr,0.0041,0.0040,helps
53,False,logreg,ALL,add_to_base,pr,0.0034,0.0079,no clear effect
71,False,logreg,target,add_to_base,pr,0.0009,0.0006,helps
67,False,logreg,freq,add_to_base,pr,0.0000,0.0004,no clear effect
55,False,logreg,ratio,add_to_base,pr,-0.0006,0.0042,no clear effect
59,False,logreg,time,add_to_base,pr,-0.0006,0.0017,no clear effect
63,False,logreg,history,add_to_base,pr,-0.0006,0.0006,hurts
77,False,logreg,bins,drop_from_full,pr,0.0050,0.0034,helps
73,False,logreg,target,drop_from_full,pr,0.0009,0.0005,helps
69,False,logreg,freq,drop_from_full,pr,-0.0002,0.0002,no clear effect


## Conclusion

The engineered features did not give a clear, repeatable lift. All PR-AUC changes are at most about 1.5 points and most are smaller than the fold-to-fold std, so I do not claim a gain.

- **All groups together** vs base: PR-AUC +0.0034 (logreg) and +0.0103 (RF) without error_code; -0.0057 and +0.0095 with it. Every value is within one std (0.008-0.015): no clear effect.
- **Bins** is the only group with a consistent sign in both tests: logreg without error_code, +0.0041 (add) and +0.0050 (drop), std 0.003-0.004. The gain is small and disappears when error_code is present, so it is a weak finding. For RF, bins is negative or inconsistent.
- **Freq and target encoding** show isolated "helps" labels (RF freq +0.0151 with error_code, RF target +0.0052 without), but the matching drop test disagrees or is within noise. I treat these as noise.
- **History** features, which I expected to help most, show no gain (slightly negative for logreg, about zero for RF).
- **error_code** dominates: with it present, group effects are even less clear.

Caveats: 5 folds and about 3.9% positives make per-fold PR-AUC noisy. The "mean > std" rule is lenient, and with 52 PR-AUC comparisons several flags are expected by chance, so only results that agree between add-to-base and drop-from-full are treated as evidence. The ablation covers logreg and RF only; tree boosting (Day 9) may use these features differently.

Decision: keep the feature code and tests (the pipeline is leak-free and ready), but do not present the features as a proven improvement. Day 9 will choose the final feature set by CV on the boosted models.

In [9]:
# setup
from scipy.stats import loguniform, randint, uniform  # noqa: E402
from sklearn.model_selection import RandomizedSearchCV  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402

# Day 7 best baseline (Logistic Regression), 5-fold mean +- std
BASELINE = {"model": "Logistic Regression (L2)", "pr_auc": 0.2221, "roc_auc": 0.8092}

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
SPW = n_neg / n_pos
# Full balancing often hurts PR-AUC, so let the search choose the strength.
SPW_CHOICES = [1.0, round(SPW**0.5, 2), round(SPW, 2)]
print(n_pos, n_neg, SPW_CHOICES)


def feature_kwargs(name):
    """Pipeline arguments for one feature set: base, stateless, or all."""
    if name == "base":
        return {}
    kw = {"num_cols": NUMERIC_FEATURES + ENGINEERED}
    if name == "all":
        kw.update(
            freq_cols=["account_id"],
            target_cols=["account_id"],
            bin_cols=["latency_ms", "retry_count"],
        )
    return kw


FEATURE_SETS = ["base", "stateless", "all"]


def run_search(model, space, set_name, n_iter=25):
    """Randomized search over the whole pipeline, scored on PR-AUC."""
    pipe = build_pipeline(model, **feature_kwargs(set_name))
    search = RandomizedSearchCV(
        pipe,
        {f"clf__{k}": v for k, v in space.items()},
        n_iter=n_iter,
        scoring=SCORING,
        refit="pr",
        cv=cv,
        random_state=SEED,
        n_jobs=-1,
    )
    return search.fit(x_train, y_train)


def summarize(searches, label):
    """Best CV result per feature set."""
    rows = []
    for name, s in searches.items():
        i, r = s.best_index_, s.cv_results_
        rows.append(
            {
                "model": label,
                "features": name,
                "pr_auc": r["mean_test_pr"][i],
                "pr_std": r["std_test_pr"][i],
                "roc_auc": r["mean_test_roc"][i],
                "roc_std": r["std_test_roc"][i],
            }
        )
    return pd.DataFrame(rows).round(4)

685 16815 [1.0, 4.95, 24.55]


In [10]:
#run the XGBoost search
XGB_SPACE = {
    "n_estimators": randint(100, 500),
    "learning_rate": loguniform(0.01, 0.2),
    "max_depth": randint(2, 7),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "reg_lambda": loguniform(0.1, 20),
    "reg_alpha": loguniform(1e-3, 5),
    "scale_pos_weight": SPW_CHOICES,
}
xgb_searches = {}
for fs in FEATURE_SETS:
    model = XGBClassifier(
        tree_method="hist", eval_metric="logloss", random_state=SEED, n_jobs=1
    )
    xgb_searches[fs] = run_search(model, XGB_SPACE, fs)
    print(fs, "done")

base done
stateless done
all done


In [11]:
#results
xgb_summary = summarize(xgb_searches, "xgboost")
print(f"Baseline PR-AUC {BASELINE['pr_auc']}, ROC-AUC {BASELINE['roc_auc']}")
display(xgb_summary)
for fs, s in xgb_searches.items():
    print(fs, s.best_params_)

Baseline PR-AUC 0.2221, ROC-AUC 0.8092


,model,features,pr_auc,pr_std,roc_auc,roc_std
0,xgboost,base,0.2715,0.0286,0.8172,0.0080
1,xgboost,stateless,0.2698,0.0312,0.8143,0.0066
2,xgboost,all,0.2688,0.0257,0.8181,0.0112


base {'clf__colsample_bytree': np.float64(0.610167650697638), 'clf__learning_rate': np.float64(0.013815607382950859), 'clf__max_depth': 2, 'clf__min_child_weight': 4, 'clf__n_estimators': 195, 'clf__reg_alpha': np.float64(0.3738463468988909), 'clf__reg_lambda': np.float64(0.2092212805678861), 'clf__scale_pos_weight': 24.55, 'clf__subsample': np.float64(0.7641531692142519)}
stateless {'clf__colsample_bytree': np.float64(0.610167650697638), 'clf__learning_rate': np.float64(0.013815607382950859), 'clf__max_depth': 2, 'clf__min_child_weight': 4, 'clf__n_estimators': 195, 'clf__reg_alpha': np.float64(0.3738463468988909), 'clf__reg_lambda': np.float64(0.2092212805678861), 'clf__scale_pos_weight': 24.55, 'clf__subsample': np.float64(0.7641531692142519)}
all {'clf__colsample_bytree': np.float64(0.7272013899887455), 'clf__learning_rate': np.float64(0.013905315749737282), 'clf__max_depth': 2, 'clf__min_child_weight': 9, 'clf__n_estimators': 215, 'clf__reg_alpha': np.float64(0.020143810352642465)

In [12]:
#run the LightGBM search
from lightgbm import LGBMClassifier  # noqa: E402

LGBM_SPACE = {
    "n_estimators": randint(100, 500),
    "learning_rate": loguniform(0.01, 0.2),
    "num_leaves": randint(3, 16),
    "max_depth": randint(2, 6),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_samples": randint(10, 60),
    "reg_lambda": loguniform(0.1, 20),
    "reg_alpha": loguniform(1e-3, 5),
    "scale_pos_weight": SPW_CHOICES,
}
lgbm_searches = {}
for fs in FEATURE_SETS:
    model = LGBMClassifier(
        subsample_freq=1, random_state=SEED, n_jobs=1, verbose=-1
    )
    lgbm_searches[fs] = run_search(model, LGBM_SPACE, fs)
    print(fs, "done")

base done
stateless done
all done


In [13]:
#results
lgbm_summary = summarize(lgbm_searches, "lightgbm")
print(f"Baseline PR-AUC {BASELINE['pr_auc']}, ROC-AUC {BASELINE['roc_auc']}")
display(pd.concat([xgb_summary, lgbm_summary], ignore_index=True))
for fs, s in lgbm_searches.items():
    print(fs, {k: round(float(v), 4) for k, v in s.best_params_.items()})

Baseline PR-AUC 0.2221, ROC-AUC 0.8092


,model,features,pr_auc,pr_std,roc_auc,roc_std
0,xgboost,base,0.2715,0.0286,0.8172,0.0080
1,xgboost,stateless,0.2698,0.0312,0.8143,0.0066
2,xgboost,all,0.2688,0.0257,0.8181,0.0112
3,lightgbm,base,0.2689,0.0339,0.8167,0.0071
4,lightgbm,stateless,0.2705,0.0270,0.8129,0.0112
5,lightgbm,all,0.2709,0.0293,0.8126,0.0135


base {'clf__colsample_bytree': 0.7335, 'clf__learning_rate': 0.0153, 'clf__max_depth': 4.0, 'clf__min_child_samples': 31.0, 'clf__n_estimators': 408.0, 'clf__num_leaves': 4.0, 'clf__reg_alpha': 0.4684, 'clf__reg_lambda': 14.4424, 'clf__scale_pos_weight': 4.95, 'clf__subsample': 0.6727}
stateless {'clf__colsample_bytree': 0.7555, 'clf__learning_rate': 0.0225, 'clf__max_depth': 2.0, 'clf__min_child_samples': 33.0, 'clf__n_estimators': 316.0, 'clf__num_leaves': 14.0, 'clf__reg_alpha': 0.176, 'clf__reg_lambda': 0.4316, 'clf__scale_pos_weight': 1.0, 'clf__subsample': 0.9209}
all {'clf__colsample_bytree': 0.7555, 'clf__learning_rate': 0.0225, 'clf__max_depth': 2.0, 'clf__min_child_samples': 33.0, 'clf__n_estimators': 316.0, 'clf__num_leaves': 14.0, 'clf__reg_alpha': 0.176, 'clf__reg_lambda': 0.4316, 'clf__scale_pos_weight': 1.0, 'clf__subsample': 0.9209}


In [14]:
#Scores the baseline, best XGBoost and best LightGBM on 15 fresh folds (5×3, new seed) that the search never saw, so the comparison is fair.
from scipy import stats  # noqa: E402
from sklearn.base import clone  # noqa: E402
from sklearn.model_selection import RepeatedStratifiedKFold  # noqa: E402

cv_final = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2026)

candidates = {
    "baseline_logreg": build_pipeline(LogisticRegression(max_iter=1000, random_state=SEED)),
    "xgb_base": clone(xgb_searches["base"].best_estimator_),
    "lgbm_base": clone(lgbm_searches["base"].best_estimator_),
}
final = {}
for name, pipe in candidates.items():
    res = cross_validate(pipe, x_train, y_train, cv=cv_final, scoring=SCORING, n_jobs=-1)
    final[name] = {"pr": res["test_pr"], "roc": res["test_roc"]}
    print(name, "done")

baseline_logreg done
xgb_base done
lgbm_base done


In [15]:
#Computes each model’s mean with a 95% confidence interval, tests whether the tuned models beat the baseline (paired differences and p-values), and saves both tables to reports/.
K = 15
TEST_OVER_TRAIN = 1 / 4  # 5-fold: each test fold is 1/4 the size of the train part
T_CRIT = stats.t.ppf(0.975, K - 1)


def corrected_se(values):
    """Std error of a mean over overlapping CV folds (Nadeau-Bengio)."""
    return np.sqrt((1 / K + TEST_OVER_TRAIN) * values.var(ddof=1))


rows = []
for name, scores in final.items():
    for metric in ("pr", "roc"):
        v = scores[metric]
        half = T_CRIT * corrected_se(v)
        rows.append(
            {
                "model": name,
                "metric": metric,
                "mean": v.mean(),
                "fold_std": v.std(ddof=1),
                "ci95_low": v.mean() - half,
                "ci95_high": v.mean() + half,
            }
        )
final_table = pd.DataFrame(rows).round(4)
display(final_table)

pairs = [
    ("xgb_base", "baseline_logreg"),
    ("lgbm_base", "baseline_logreg"),
    ("xgb_base", "lgbm_base"),
]
prow = []
for a, b in pairs:
    for metric in ("pr", "roc"):
        d = final[a][metric] - final[b][metric]
        se = corrected_se(d)
        t = d.mean() / se
        half = T_CRIT * se
        prow.append(
            {
                "comparison": f"{a} - {b}",
                "metric": metric,
                "mean_diff": d.mean(),
                "ci95_low": d.mean() - half,
                "ci95_high": d.mean() + half,
                "p_value": 2 * stats.t.sf(abs(t), K - 1),
            }
        )
paired_table = pd.DataFrame(prow).round(4)
display(paired_table)
os.makedirs("reports", exist_ok=True)
final_table.to_csv("reports/final_cv_table.csv", index=False)
paired_table.to_csv("reports/final_paired_table.csv", index=False)

,model,metric,mean,fold_std,ci95_low,ci95_high
0,baseline_logreg,pr,0.2203,0.0345,0.1786,0.2619
1,baseline_logreg,roc,0.8067,0.0191,0.7837,0.8297
2,xgb_base,pr,0.2656,0.0444,0.2120,0.3192
3,xgb_base,roc,0.8131,0.0177,0.7917,0.8344
4,lgbm_base,pr,0.2663,0.0415,0.2163,0.3164
5,lgbm_base,roc,0.8140,0.0195,0.7905,0.8374


,comparison,metric,mean_diff,ci95_low,ci95_high,p_value
0,xgb_base - baseline_logreg,pr,0.0454,0.0195,0.0712,0.0021
1,xgb_base - baseline_logreg,roc,0.0063,-0.0033,0.0160,0.1787
2,lgbm_base - baseline_logreg,pr,0.0460,0.0226,0.0694,0.0009
3,lgbm_base - baseline_logreg,roc,0.0072,-0.0018,0.0162,0.1069
4,xgb_base - lgbm_base,pr,-0.0007,-0.0170,0.0156,0.9297
5,xgb_base - lgbm_base,roc,-0.0009,-0.0076,0.0059,0.7819


In [16]:
#refit the chosen XGBoost on base features, check it on the validation split, save it.
import json  # noqa: E402
import platform  # noqa: E402

import joblib  # noqa: E402
import sklearn  # noqa: E402
import xgboost  # noqa: E402
from sklearn.metrics import average_precision_score, roc_auc_score  # noqa: E402

x_base, y_base = split_xy(train)
best_model = clone(xgb_searches["base"].best_estimator_).fit(x_base, y_base)

val_x, val_y = split_xy(_val)
val_proba = best_model.predict_proba(val_x)[:, 1]
print("validation PR-AUC ", round(average_precision_score(val_y, val_proba), 4))
print("validation ROC-AUC", round(roc_auc_score(val_y, val_proba), 4))

os.makedirs("models", exist_ok=True)
joblib.dump(best_model, "models/best_model.joblib")
np.save("models/reference_proba.npy", val_proba[:200])
meta = {
    "model": "XGBClassifier, base features",
    "input_columns": list(x_base.columns),
    "params": {k: float(v) if hasattr(v, "item") else v
               for k, v in xgb_searches["base"].best_params_.items()},
    "python": platform.python_version(),
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}
with open("models/model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("saved")

validation PR-AUC  0.2459
validation ROC-AUC 0.793
saved
